# Analysis & Visualization

1. UMAP: Catch22 features vs MOMENT embeddings
2. Full results table (all models)
3. Confusion matrix comparison: best baseline vs best FM
4. Ablation: adaptation strategy, class imbalance handling

In [1]:
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import umap
from sklearn.preprocessing import StandardScaler
import pycatch22
import itertools

from src.data_loader import load_lsst_for_sklearn, SEED
from src.metrics import plot_confusion_matrix, results_table

In [2]:
data = load_lsst_for_sklearn()
X_train = data["X_train"]  # (n, 6, 36)
y_train = data["y_train"]
class_names = data["class_names"]

# Load saved MOMENT embeddings
saved = np.load("moment_embeddings.npz")
emb_train = saved["emb_train"]
print(f"Catch22 input: {X_train.shape}, MOMENT emb: {emb_train.shape}")

FileNotFoundError: [Errno 2] No such file or directory: 'moment_embeddings.npz'

## 1. UMAP: Catch22 vs MOMENT Embeddings

In [ ]:
# Compute Catch22 features (same as Baseline notebook)
def compute_catch22_features(X_cf):
    """X_cf: (n, C, T) channels-first."""
    n, c, t = X_cf.shape
    all_feats = []
    for i in range(n):
        sample_feats = []
        for ch in range(c):
            ts = X_cf[i, ch, :]
            result = pycatch22.catch22_all(ts)
            sample_feats += result["values"]
        all_feats.append(sample_feats)
    return np.array(all_feats)

catch22_feats = compute_catch22_features(X_train)
catch22_feats = np.nan_to_num(catch22_feats, nan=0.0)
print(f"Catch22 features: {catch22_feats.shape}")

In [ ]:
# UMAP projections
scaler_c22 = StandardScaler()
catch22_scaled = scaler_c22.fit_transform(catch22_feats)

scaler_emb = StandardScaler()
emb_scaled = scaler_emb.fit_transform(emb_train)

umap_c22 = umap.UMAP(n_components=2, random_state=SEED).fit_transform(catch22_scaled)
umap_emb = umap.UMAP(n_components=2, random_state=SEED).fit_transform(emb_scaled)

In [ ]:
fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(12, 5), dpi=150)

for ax, proj, title in [
    (ax1, umap_c22, "Catch22 Features"),
    (ax2, umap_emb, "MOMENT Embeddings"),
]:
    sc = ax.scatter(
        proj[:, 0], proj[:, 1],
        c=y_train, cmap="tab20", s=8, alpha=0.7, linewidths=0,
    )
    ax.set_title(title, fontsize=12)
    ax.set_xlabel("UMAP-1")
    ax.set_ylabel("UMAP-2")
    ax.spines["top"].set_visible(False)
    ax.spines["right"].set_visible(False)

# Shared legend
handles, _ = sc.legend_elements()
fig.legend(handles, class_names, title="Class", loc="center right",
           fontsize=8, title_fontsize=9, frameon=False,
           bbox_to_anchor=(1.12, 0.5))

plt.tight_layout()
plt.savefig("figures/umap_comparison.pdf", format="pdf", bbox_inches="tight")
plt.show()

## 2. Full Results Table

Paste your metrics from notebooks 02 and 03 here.

In [ ]:
# Fill in with actual numbers after running experiments
# These are placeholder values — update after execution
all_results = {
    "Catch22 + RF (B1)":       dict(accuracy=0.60, weighted_f1=0.54, macro_f1=0.31),
    "MultiROCKET (B2)":        dict(accuracy=0.00, weighted_f1=0.00, macro_f1=0.00),  # TODO
    "1D-CNN (B3)":             dict(accuracy=0.00, weighted_f1=0.00, macro_f1=0.00),  # TODO
    "MOMENT ZS + LogReg (M1)": dict(accuracy=0.00, weighted_f1=0.00, macro_f1=0.00),  # TODO
    "MOMENT ZS + RF (M1)":     dict(accuracy=0.00, weighted_f1=0.00, macro_f1=0.00),  # TODO
    "MOMENT Head-FT (M2)":     dict(accuracy=0.00, weighted_f1=0.00, macro_f1=0.00),  # TODO
    "MOMENT Full-FT (M3)":     dict(accuracy=0.00, weighted_f1=0.00, macro_f1=0.00),  # TODO
}
results_table(all_results)

## 3. Confusion Matrix Comparison

In [ ]:
# Load predictions from notebooks 02 and 03 or re-run models
# Example: compare best baseline vs best FM
# Uncomment and update once you have predictions saved:
#
# fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(16, 7))
# plot_confusion_matrix(y_test, y_pred_best_baseline, class_names,
#                       title="Best Baseline", ax=ax1)
# plot_confusion_matrix(y_test, y_pred_best_fm, class_names,
#                       title="Best Foundation Model", ax=ax2)
# plt.tight_layout()
# plt.savefig("figures/confusion_comparison.pdf", bbox_inches="tight")
# plt.show()

print("Run experiments in notebooks 02 and 03 first, then update this cell.")

## 4. Ablation: Adaptation Strategy

Compare zero-shot vs head-only vs full fine-tuning on weighted F1.

In [ ]:
# Ablation bar chart — update values after running experiments
strategies = ["Zero-shot\n+ LogReg", "Zero-shot\n+ RF", "Head-only\nFine-tune", "Full\nFine-tune"]
wf1_scores = [0.0, 0.0, 0.0, 0.0]  # TODO: fill in

fig, ax = plt.subplots(figsize=(6, 4), dpi=150)
bars = ax.bar(strategies, wf1_scores, color=["#4C72B0", "#55A868", "#C44E52", "#8172B2"])
ax.set_ylabel("Weighted F1")
ax.set_title("MOMENT: Effect of Adaptation Strategy")
ax.set_ylim(0, 1)
for bar, score in zip(bars, wf1_scores):
    ax.text(bar.get_x() + bar.get_width() / 2, bar.get_height() + 0.02,
            f"{score:.3f}", ha="center", va="bottom", fontsize=10)
ax.spines["top"].set_visible(False)
ax.spines["right"].set_visible(False)
plt.tight_layout()
# plt.savefig("figures/ablation_strategy.pdf", bbox_inches="tight")
plt.show()